In [1]:
import pandas as pd
from demandfcst import config, storage
from demandfcst.pipeline import schema

loc = config.locations()
history = storage.join(loc["landing"], "sellin_daily", "ingest_date=2024-12-29", "part.csv")
daily = schema.read_csv(history, schema.SCHEMAS["sellin_daily"])
print(daily.shape)
print(daily.dtypes)
daily.head(3)


(223125, 4)
date         datetime64[us]
province                str
sku_id                  str
volume_hl           float64
dtype: object


,date,province,sku_id,volume_hl
0,2021-01-01,BA,18747,2.100
1,2021-01-01,BA,22575,2.692
2,2021-01-01,BA,27988,4.774


In [2]:
skus = schema.read_csv(storage.join(loc["landing"], "sku_master", "ingest_date=2024-12-29", "part.csv"),
                       schema.SCHEMAS["sku_master"])
print(skus.dtypes)
skus[skus["launch_date"].notna() | skus["discontinue_date"].notna()][["sku_id", "sku_name", "launch_date", "discontinue_date"]]


sku_id                         str
sku_name                       str
brand                          str
pack                           str
container                      str
size_ml                      Int64
units_per_case               Int64
returnable                 boolean
launch_date         datetime64[us]
discontinue_date    datetime64[us]
dtype: object


,sku_id,sku_name,launch_date,discontinue_date
15,73000,BUDWEISER LATA 269ML,2023-06-05,NaT
16,32093,BUDWEISER LATA 350ML,2024-04-01,NaT
17,24456,BUDWEISER LATA 473ML,NaT,2022-12-25
24,51300,ORIGINAL LONG NECK 330ML,2022-03-07,NaT
27,90806,STELLA ARTOIS LATA 350ML,NaT,2023-08-27
35,35942,SPATEN LATA 269ML,2022-10-03,NaT
38,31554,SPATEN GARRAFA 600ML,NaT,2024-06-30


In [3]:
bad = pd.DataFrame({"date": ["2024-01-01", "01/02/2024"], "province": ["SP", None],
                    "sku_id": ["00123", "38958"], "volume_hl": ["1.5", "abc"]})
try:
    schema.coerce(bad, schema.SCHEMAS["sellin_daily"])
except schema.SchemaError as e:
    for p in e.problems:
        print("-", p)


- date: 1 value(s) are not date, e.g. ['01/02/2024']
- province: 1 empty value(s)
- volume_hl: 1 value(s) are not float, e.g. ['abc']


In [1]:
from demandfcst import config
from demandfcst.pipeline import reader

raw = config.pipeline_paths()["raw"]

sales = reader.read_dataset(raw, "sellin_daily")
print(sales.shape)
sales.groupby("ingest_date").agg(rows=("volume_hl", "size"),
                                 first_day=("date", "min"),
                                 last_day=("date", "max"),
                                 total_hl=("volume_hl", "sum")).round(0)


(231859, 5)


C:\Users\haris\AppData\Local\Temp\ipykernel_32480\1963499026.py:11: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  total_hl=("volume_hl", "sum")).round(0)


,rows,first_day,last_day,total_hl
ingest_date,,,,
2024-12-29,223125,2021-01-01,2024-12-28,8983295.0
2025-01-05,1095,2024-12-30,2025-01-04,59486.0
2025-01-12,1096,2025-01-06,2025-01-11,48505.0
2025-01-19,1087,2025-01-13,2025-01-18,56121.0
2025-01-26,1091,2025-01-20,2025-01-25,55093.0
2025-02-02,1091,2025-01-27,2025-02-01,56424.0
2025-02-09,1094,2025-02-03,2025-02-08,59362.0
2025-02-16,1087,2025-02-10,2025-02-15,54581.0
2025-02-23,1093,2025-02-17,2025-02-22,65312.0


In [2]:
wx = reader.read_dataset(raw, "weather_weekly")
skus = reader.read_latest(raw, "sku_master")
provinces = reader.read_latest(raw, "province_master")
print("weather:", wx.shape, "| skus:", skus.shape, "| provinces:", provinces.shape)
print("master snapshot from:", skus["ingest_date"].iloc[0].date())


weather: (1953, 7) | skus: (40, 11) | provinces: (9, 4)
master snapshot from: 2024-12-29
